# 03 — Traditional ML Baseline

TF-IDF + classifier. Cepat di CPU (~30-60 menit), tidak butuh GPU.
Target: balanced accuracy **≥ 0.65** sebelum lanjut ke BERT.

In [1]:
# !pip install -q scikit-learn imbalanced-learn xgboost lightgbm openpyxl pandas
import re, warnings, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
warnings.filterwarnings('ignore')

try:
    from lightgbm import LGBMClassifier; HAS_LGBM = True
except ImportError:
    HAS_LGBM = False; print('LGBM not installed')

DATA_DIR   = Path('../data')
OUTPUT_DIR = DATA_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_ORDER = [
    'Anggaran', 'Distribusi', 'Ekonomi', 'Kualitas Pangan',
    'Lainnya', 'Politik', 'Sasaran Penerima', 'Tata Kelola',
]
SEED = 42

In [2]:
def clean_text(text):
    text = str(text).encode('utf-8','ignore').decode('utf-8')
    text = re.sub(r'http\S+|www\.\S+', '[URL]', text)
    text = re.sub(r'@\w+', '[USER]', text)
    text = re.sub(r'^\s*RT\s+', '', text.strip())
    text = re.sub(r'([!?.]){2,}', r'\1', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    return re.sub(r'\s+', ' ', text).strip()

def make_sampling_dict(labels, ratio):
    counts  = Counter(labels.tolist())
    target  = int(max(counts.values()) * ratio)
    return {cls: target for cls, n in counts.items() if n < target}

# Gunakan LLM-labeled jika ada, fallback ke original
try:
    labeled = pd.read_excel(DATA_DIR / 'labeled/hasil_label_baru.xlsx')
    print('Using LLM-labeled data')
except FileNotFoundError:
    labeled = pd.read_excel(DATA_DIR / 'raw/case_1_labeled_data.xlsx')
    print('Using original labels')

to_predict = pd.read_excel(DATA_DIR / 'raw/case_1_text_to_predict.xlsx')
template   = pd.read_excel(DATA_DIR / 'raw/case_1_template_sheet.xlsx')
for df in [labeled, to_predict, template]:
    df.columns = [c.strip().lower() for c in df.columns]

labeled['label'] = labeled['label'].astype(str).str.strip()
labeled['clean'] = labeled['full_text'].apply(clean_text)
to_predict['clean'] = to_predict['full_text'].apply(clean_text)

le = LabelEncoder(); le.fit(LABEL_ORDER)
labeled['label_enc'] = le.transform(labeled['label'])
texts  = labeled['clean'].tolist(); labels = labeled['label_enc'].values
print(f'Train={len(texts)}  Test={len(to_predict)}')

Using LLM-labeled data
Train=5000  Test=1500


## Experiment Grid

In [3]:
def get_clf(name):
    if name == 'svm':
        return CalibratedClassifierCV(
            LinearSVC(C=1.0, class_weight='balanced', max_iter=3000, random_state=SEED),
            cv=3, method='isotonic')
    if name == 'logistic':
        return LogisticRegression(C=1.0, class_weight='balanced',
                                   max_iter=1000, solver='lbfgs', random_state=SEED)
    if name == 'lgbm' and HAS_LGBM:
        return LGBMClassifier(n_estimators=300, learning_rate=0.1, class_weight='balanced',
                               num_leaves=63, random_state=SEED, n_jobs=-1, verbose=-1)
    return None

def run_exp(clf_name, strategy, ratio):
    clf = get_clf(clf_name)
    if clf is None: return None

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=50000, sublinear_tf=True, min_df=2)
    oof_preds = np.zeros(len(texts), dtype=np.int64)

    for tr_idx, val_idx in skf.split(texts, labels):
        tr_t  = [texts[i] for i in tr_idx]; val_t = [texts[i] for i in val_idx]
        tr_l  = labels[tr_idx].copy()
        X_tr  = tfidf.fit_transform(tr_t); X_val = tfidf.transform(val_t)

        if strategy == 'smote':
            ss = make_sampling_dict(tr_l, ratio)
            try: X_tr, tr_l = SMOTE(random_state=SEED, sampling_strategy=ss, k_neighbors=5).fit_resample(X_tr, tr_l)
            except Exception: pass
        elif strategy == 'smote_tomek':
            ss = make_sampling_dict(tr_l, ratio)
            try: X_tr, tr_l = SMOTETomek(random_state=SEED, smote=SMOTE(random_state=SEED, sampling_strategy=ss, k_neighbors=5)).fit_resample(X_tr, tr_l)
            except Exception: pass

        clf.fit(X_tr, tr_l)
        oof_preds[val_idx] = clf.predict(X_val)

    bal = balanced_accuracy_score(labels, oof_preds)
    rpt = classification_report(labels, oof_preds, target_names=le.classes_, output_dict=True, zero_division=0)
    return {'model':clf_name,'strategy':strategy,'ratio':ratio,'bal_acc':round(bal,4),
            'ekonomi_recall':round(rpt.get('Ekonomi',{}).get('recall',0),4)}

GRID = [
    ('svm',     'none',        1.0),
    ('logistic','none',        1.0),
    ('lgbm',    'none',        1.0),
    ('svm',     'smote',       0.8),
    ('lgbm',    'smote',       0.8),
    ('lgbm',    'smote_tomek', 1.0),
]

results = []
for clf_name, strategy, ratio in GRID:
    t0 = time.time()
    print(f'{clf_name:<10} {strategy:<12} r={ratio}', end='  ')
    r = run_exp(clf_name, strategy, ratio)
    if r:
        results.append(r)
        print(f'bal_acc={r["bal_acc"]:.4f}  ekonomi={r["ekonomi_recall"]:.4f}  ({time.time()-t0:.0f}s)')
    else:
        print('SKIPPED')

svm        none         r=1.0  bal_acc=0.6365  ekonomi=0.6020  (2s)
logistic   none         r=1.0  bal_acc=0.6418  ekonomi=0.6429  (4s)
lgbm       none         r=1.0  bal_acc=0.5576  ekonomi=0.4847  (64s)
svm        smote        r=0.8  bal_acc=0.5632  ekonomi=0.3469  (4s)
lgbm       smote        r=0.8  bal_acc=0.5942  ekonomi=0.5306  (138s)
lgbm       smote_tomek  r=1.0  bal_acc=0.6042  ekonomi=0.5255  (181s)


In [4]:
results_df = pd.DataFrame(results).sort_values('bal_acc', ascending=False).reset_index(drop=True)
print('\n=== RESULTS ===')
print(results_df[['model','strategy','ratio','bal_acc','ekonomi_recall']].to_string(index=True))
results_df.to_csv(OUTPUT_DIR / 'baseline_results.csv', index=False)

best = results_df.iloc[0]
print(f'\nBest: {best["model"]} | {best["strategy"]} | r={best["ratio"]} | bal_acc={best["bal_acc"]:.4f}')


=== RESULTS ===
      model     strategy  ratio  bal_acc  ekonomi_recall
0  logistic         none    1.0   0.6418          0.6429
1       svm         none    1.0   0.6365          0.6020
2      lgbm  smote_tomek    1.0   0.6042          0.5255
3      lgbm        smote    0.8   0.5942          0.5306
4       svm        smote    0.8   0.5632          0.3469
5      lgbm         none    1.0   0.5576          0.4847

Best: logistic | none | r=1.0 | bal_acc=0.6418


## Generate Baseline Submission

In [5]:
from collections import Counter as _Counter

def make_sampling_dict_full(labels, ratio):
    counts = _Counter(labels.tolist())
    target = int(max(counts.values()) * ratio)
    return {cls: target for cls, n in counts.items() if n < target}

tfidf_full = TfidfVectorizer(ngram_range=(1,2), max_features=50000, sublinear_tf=True, min_df=2)
X_train_full = tfidf_full.fit_transform(texts)
X_test_full  = tfidf_full.transform(to_predict['clean'].tolist())
tr_labels_full = labels.copy()

if best['strategy'] == 'smote':
    ss = make_sampling_dict_full(tr_labels_full, best['ratio'])
    try: X_train_full, tr_labels_full = SMOTE(random_state=SEED, sampling_strategy=ss, k_neighbors=5).fit_resample(X_train_full, tr_labels_full)
    except Exception as e: print(f'Resample fail: {e}')

clf_final = get_clf(best['model'])
clf_final.fit(X_train_full, tr_labels_full)
preds_str = le.inverse_transform(clf_final.predict(X_test_full))

sub = template.copy()
sub['label'] = preds_str
sub_path = OUTPUT_DIR / 'submissions/submission_baseline.xlsx'
sub_path.parent.mkdir(exist_ok=True)
sub.to_excel(str(sub_path), index=False)
print(f'Saved: {sub_path}')
print(f'OOF balanced accuracy: {best["bal_acc"]:.4f}')
print('\nDistribusi prediksi:')
for lbl, cnt in pd.Series(preds_str).value_counts().items():
    print(f'  {lbl:<22} {cnt:>4}  ({cnt/len(preds_str)*100:.1f}%)')

Saved: ..\data\outputs\submissions\submission_baseline.xlsx
OOF balanced accuracy: 0.6418

Distribusi prediksi:
  Kualitas Pangan         278  (18.5%)
  Anggaran                249  (16.6%)
  Lainnya                 223  (14.9%)
  Politik                 206  (13.7%)
  Sasaran Penerima        171  (11.4%)
  Distribusi              157  (10.5%)
  Tata Kelola             148  (9.9%)
  Ekonomi                  68  (4.5%)
